# Programación funcional · Comprensiones y orden superior

Problemas resueltos sin bucles explícitos: comprensiones, generadores infinitos con `itertools.count`, `takewhile` y `next`, funciones que reciben funciones (`segmentos`, `relacionados`, `filtra_aplica`), `zip(*m)`, `pairwise`, `groupby` y `chain`. Cada solución se comprueba por otra vía: una fórmula cerrada, la fuerza bruta o una versión iterativa. Ideas de los ejercicios de [José A. Alonso](https://github.com/jaalonso/Ejercicios-Python) y de [Project Euler](https://projecteuler.net/archives) (problemas 1, 6 y 9).

## 1. Múltiplos de 3 o de 5

Calcular con una comprensión la suma de los naturales menores que 1000 que son múltiplos de 3 o de 5. Comprobarlo con la fórmula de la suma de una progresión aritmética y el principio de inclusión-exclusión, $S(3) + S(5) - S(15)$, que sirve también para límites enormes como $10^{18}$.

In [1]:
def suma_multiplos(n):
    return sum(x for x in range(n) if x % 3 == 0 or x % 5 == 0)


def suma_multiplos_de(k, n):
    m = (n - 1) // k
    return k * m * (m + 1) // 2


def suma_multiplos_formula(n):
    return suma_multiplos_de(3, n) + suma_multiplos_de(5, n) - suma_multiplos_de(15, n)


print("Menores que 1000:", suma_multiplos(1000))
print("Menores que 10¹⁸:", suma_multiplos_formula(10**18))
assert all(suma_multiplos(n) == suma_multiplos_formula(n) for n in range(1, 2000))

Menores que 1000: 233168
Menores que 10¹⁸: 233333333333333333166666666666666668


## 2. Cuadrado de la suma frente a suma de cuadrados

Calcular con comprensiones la diferencia entre el cuadrado de la suma de los $n$ primeros naturales y la suma de sus cuadrados para $n = 10$ y $n = 100$. Comprobarlo con las fórmulas $\sum k = \frac{n(n+1)}{2}$ y $\sum k^2 = \frac{n(n+1)(2n+1)}{6}$.

In [2]:
def diferencia(n):
    return sum(range(1, n + 1)) ** 2 - sum(k * k for k in range(1, n + 1))


def diferencia_formula(n):
    return (n * (n + 1) // 2) ** 2 - n * (n + 1) * (2 * n + 1) // 6


for n in (10, 100):
    print(f"n = {n:>3}: {diferencia(n)}")
assert all(diferencia(n) == diferencia_formula(n) for n in range(1, 500))

n =  10: 2640
n = 100: 25164150


## 3. Terna pitagórica que suma 1000

Encontrar con una comprensión y `next` la única terna pitagórica $a < b < c$ con $a + b + c = 1000$, recorriendo solo $a$ y $b$ (pues $c = 1000 - a - b$). Comprobarla con la parametrización de Euclides: toda terna es $k(m^2 - n^2),\ k \cdot 2mn,\ k(m^2 + n^2)$ con $m > n > 0$.

In [3]:
def terna(perimetro):
    return next(
        ((a, b, perimetro - a - b)
         for a in range(1, perimetro // 3)
         for b in range(a + 1, (perimetro - a) // 2 + 1)
         if a * a + b * b == (perimetro - a - b) ** 2),
        None,
    )


def ternas_euclides(perimetro):
    return {
        tuple(sorted((k * (m * m - n * n), k * 2 * m * n))) + (k * (m * m + n * n),)
        for m in range(2, perimetro)
        for n in range(1, m)
        if 2 * m * (m + n) <= perimetro
        for k in [perimetro // (2 * m * (m + n))]
        if k * 2 * m * (m + n) == perimetro
    }


a, b, c = terna(1000)
print(f"{a}² + {b}² = {c}²  →  producto a·b·c = {a * b * c}")
assert ternas_euclides(1000) == {(a, b, c)}
assert terna(12) == (3, 4, 5) and terna(13) is None

200² + 375² = 425²  →  producto a·b·c = 31875000


## 4. ¿Cuántos términos hacen falta?

Sin bucles `while`, con generadores infinitos (`itertools.count`, `accumulate`) y `next`, calcular el menor número de términos que hace falta para que el error sea menor que $\varepsilon$ en tres aproximaciones:

- $e \approx \sum_{k=0}^{n-1} \frac{1}{k!}$
- $1 = \lim_{x \to 0} \frac{\sin x}{x}$, tomando $x = 1/n$
- $\pi \approx 4 \sum_{k=0}^{n-1} \frac{(-1)^k}{2k+1}$ (serie de Leibniz)

Repetir la de Leibniz con `takewhile` contando las sumas parciales que aún no llegan.

In [4]:
import math
from itertools import accumulate, count, takewhile


def primer_indice(errores, eps):
    return next(n for n, error in enumerate(errores, 1) if error < eps)


def terminos_e(eps):
    sumas = accumulate(1 / math.factorial(k) for k in count())
    return primer_indice((abs(math.e - s) for s in sumas), eps)


def terminos_seno(eps):
    return next(n for n in count(1) if abs(1 - math.sin(1 / n) * n) < eps)


def sumas_leibniz():
    return accumulate(4 * (-1) ** k / (2 * k + 1) for k in count())


def terminos_pi(eps):
    return primer_indice((abs(math.pi - s) for s in sumas_leibniz()), eps)


def terminos_pi_takewhile(eps):
    return sum(1 for _ in takewhile(lambda s: abs(math.pi - s) >= eps, sumas_leibniz())) + 1


print(f"{'ε':>7} {'e':>4} {'sen x / x':>10} {'π (Leibniz)':>12}")
for k in range(1, 6):
    eps = 10.0**-k
    print(f"{eps:>7.0e} {terminos_e(eps):>4} {terminos_seno(eps):>10} {terminos_pi(eps):>12}")
    assert terminos_pi(eps) == terminos_pi_takewhile(eps)
    # El error de Leibniz tras n términos es casi exactamente 1/n
    assert abs(terminos_pi(eps) - round(1 / eps)) <= 1
    # 1 - sen(x)/x ≈ x²/6, así que hacen falta unos 1/√(6ε) términos
    assert abs(terminos_seno(eps) - 1 / math.sqrt(6 * eps)) <= 1

      ε    e  sen x / x  π (Leibniz)
  1e-01    4          2           10
  1e-02    5          5          100
  1e-03    7         13         1000
  1e-04    8         41        10000
  1e-05    9        130       100001


## 5. Triángulo aritmético

Los naturales se colocan en filas de longitud creciente: `1`, `2 3`, `4 5 6`, `7 8 9 10`… Escribir `fila(n)` con un `range` (la fila $n$ empieza en $\frac{n(n-1)}{2} + 1$) y `triangulo(n)` con las $n$ primeras filas. Comprobar que la suma de la fila $n$ es $\frac{n(n^2+1)}{2}$ y construir también el triángulo cortando un `count(1)` con `islice`.

In [5]:
from itertools import islice


def fila(n):
    return list(range(n * (n - 1) // 2 + 1, n * (n + 1) // 2 + 1))


def triangulo(n):
    return [fila(k) for k in range(1, n + 1)]


def triangulo_islice(n):
    naturales = count(1)
    return [list(islice(naturales, k)) for k in range(1, n + 1)]


for f in triangulo(5):
    print(" ".join(f"{x:>2}" for x in f).center(16))
assert triangulo(60) == triangulo_islice(60)
assert all(sum(fila(n)) == n * (n * n + 1) // 2 for n in range(1, 300))

        1       
      2  3      
     4  5  6    
   7  8  9 10   
 11 12 13 14 15 


## 6. Segmentos maximales

Escribir `segmentos(p, xs)`, que devuelva los tramos más largos de elementos consecutivos de `xs` que cumplen el predicado `p`, con `itertools.groupby` y la clave `p`. Usarla para encontrar las rachas de días con más de 30 °C de un mes de verano y la más larga. Comprobarla con una versión con bucle.

In [6]:
import random
from itertools import groupby


def segmentos(p, xs):
    return [list(grupo) for cumple, grupo in groupby(xs, key=p) if cumple]


def segmentos_bucle(p, xs):
    resultado, actual = [], []
    for x in xs:
        if p(x):
            actual.append(x)
        elif actual:
            resultado.append(actual)
            actual = []
    return resultado + ([actual] if actual else [])


random.seed(8)
maximas = [round(random.gauss(29, 3), 1) for _ in range(31)]
rachas = segmentos(lambda t: t > 30, maximas)
print("Rachas de calor:", rachas)
print("La más larga dura", len(max(rachas, key=len)), "días")
assert segmentos(lambda x: x % 2 == 0, [1, 2, 0, 4, 1, 6, 1, 1, 8]) == [[2, 0, 4], [6], [8]]
for _ in range(500):
    xs = [random.randint(0, 9) for _ in range(random.randint(0, 20))]
    assert segmentos(lambda x: x > 4, xs) == segmentos_bucle(lambda x: x > 4, xs)

Rachas de calor: [[30.1, 36.6, 32.3, 32.3, 30.9, 30.2, 31.1], [31.0, 34.3, 30.9, 30.2, 31.3], [32.5], [38.3, 32.9, 30.6], [32.1], [32.9]]
La más larga dura 7 días


## 7. Relación entre consecutivos

Escribir `relacionados(r, xs)`, que diga si cada par de elementos consecutivos de `xs` cumple la relación binaria `r`, con `all` e `itertools.pairwise`. Usarla para saber si una lista es creciente o estrictamente creciente y si en el perfil de altitudes de un sendero ningún tramo sube o baja más de 150 m. Comprobarla con `zip(xs, xs[1:])`.

In [7]:
import operator
from itertools import pairwise


def relacionados(r, xs):
    return all(r(a, b) for a, b in pairwise(xs))


def relacionados_zip(r, xs):
    return all(r(a, b) for a, b in zip(xs, xs[1:]))


sendero = [420, 510, 640, 760, 905, 1010, 980, 1120, 1270]
print("Creciente:", relacionados(operator.le, [1, 3, 3, 7]))
print("Estrictamente creciente:", relacionados(operator.lt, [1, 3, 3, 7]))
print("Sendero suave:", relacionados(lambda a, b: abs(b - a) <= 150, sendero))
assert relacionados(operator.lt, []) and relacionados(operator.lt, [5])
random.seed(7)
for _ in range(500):
    xs = sorted(random.choices(range(10), k=random.randint(0, 8))) if random.random() < 0.5 else random.choices(range(10), k=6)
    for r in (operator.le, operator.lt, operator.ne):
        assert relacionados(r, xs) == relacionados_zip(r, xs)
    assert relacionados(operator.le, xs) == (xs == sorted(xs))

Creciente: True
Estrictamente creciente: False
Sendero suave: True


## 8. Transponer con `zip(*m)`

Transponer una matriz dada como lista de filas con `zip(*m)`, que pasa cada fila como un argumento distinto, y comprobarlo con una comprensión doble. Ver qué ocurre con filas de distinta longitud (`zip` corta, `zip_longest` rellena) y usar el mismo truco para separar una lista de pares `(isla, superficie)` en dos tuplas.

In [8]:
from itertools import zip_longest


def transpuesta(m):
    return [list(columna) for columna in zip(*m)]


def transpuesta_comprension(m):
    return [[fila[j] for fila in m] for j in range(len(m[0]))] if m else []


m = [[1, 2, 3], [4, 5, 6]]
print(transpuesta(m))
irregular = [[1, 2, 3], [4], [5, 6]]
print("zip corta:        ", transpuesta(irregular))
print("zip_longest rellena:", [list(c) for c in zip_longest(*irregular, fillvalue=0)])

superficies = [("Tenerife", 2034), ("Fuerteventura", 1660), ("Gran Canaria", 1560)]
islas, km2 = zip(*superficies)
print(islas, km2)

random.seed(1)
for _ in range(200):
    filas, columnas = random.randint(1, 5), random.randint(1, 5)
    a = [[random.randint(0, 9) for _ in range(columnas)] for _ in range(filas)]
    assert transpuesta(a) == transpuesta_comprension(a)
    assert transpuesta(transpuesta(a)) == a
assert list(zip(islas, km2)) == superficies

[[1, 4], [2, 5], [3, 6]]
zip corta:         [[1, 4, 5]]
zip_longest rellena: [[1, 4, 5], [2, 0, 6], [3, 0, 0]]
('Tenerife', 'Fuerteventura', 'Gran Canaria') (2034, 1660, 1560)


## 9. `filtra_aplica` de cinco maneras

Escribir `filtra_aplica(f, p, xs)`, que aplique `f` a los elementos de `xs` que cumplen `p`, de cinco maneras: con una comprensión, con `map` y `filter`, con un bucle, por recursión y con `functools.reduce`. Comprobar que coinciden y medir con `timeit` cuál escala peor: la recursión y el `reduce` que concatena listas copian la lista en cada paso.

In [9]:
import timeit
from functools import reduce


def fa_comprension(f, p, xs):
    return [f(x) for x in xs if p(x)]


def fa_map_filter(f, p, xs):
    return list(map(f, filter(p, xs)))


def fa_bucle(f, p, xs):
    resultado = []
    for x in xs:
        if p(x):
            resultado.append(f(x))
    return resultado


def fa_recursiva(f, p, xs):
    if not xs:
        return []
    return ([f(xs[0])] if p(xs[0]) else []) + fa_recursiva(f, p, xs[1:])


def fa_reduce(f, p, xs):
    return reduce(lambda acc, x: acc + [f(x)] if p(x) else acc, xs, [])


versiones = [fa_comprension, fa_map_filter, fa_bucle, fa_recursiva, fa_reduce]
cuadrado, es_par = (lambda x: x * x), (lambda x: x % 2 == 0)
print(fa_comprension(cuadrado, es_par, range(10)))

random.seed(3)
for _ in range(200):
    xs = [random.randint(-50, 50) for _ in range(random.randint(0, 30))]
    assert len({tuple(v(cuadrado, es_par, xs)) for v in versiones}) == 1

[0, 4, 16, 36, 64]

In [10]:
import sys


def tiempo(version, n):
    xs = list(range(n))
    return min(timeit.repeat(lambda: version(cuadrado, es_par, xs), number=5, repeat=5))


# Al multiplicar n por 16, lo lineal tarda unas 16 veces más y lo cuadrático, cientos
limite = sys.getrecursionlimit()
sys.setrecursionlimit(10_000)
crecimiento = {v.__name__: tiempo(v, 4000) / tiempo(v, 250) for v in versiones}
sys.setrecursionlimit(limite)
for nombre, c in crecimiento.items():
    print(f"{nombre:<15} {'cuadrática' if c > 40 else 'lineal'}")
assert crecimiento["fa_recursiva"] > 40 and crecimiento["fa_comprension"] < 40

fa_comprension  lineal
fa_map_filter   lineal
fa_bucle        lineal
fa_recursiva    cuadrática
fa_reduce       cuadrática


## 10. Intersección de intervalos

Dos listas de intervalos cerrados `[inicio, fin]`, cada una ordenada y sin solapamientos, guardan las horas libres de dos personas. Escribir `interseccion(a, b)`, que devuelva los intervalos en que ambas están libres recorriendo las dos listas a la vez con dos índices, en tiempo lineal. Comprobarla con la fuerza bruta sobre los conjuntos de horas (enteras) que cubre cada lista.

In [11]:
def interseccion(a, b):
    i = j = 0
    resultado = []
    while i < len(a) and j < len(b):
        inicio, fin = max(a[i][0], b[j][0]), min(a[i][1], b[j][1])
        if inicio <= fin:
            resultado.append([inicio, fin])
        if a[i][1] < b[j][1]:
            i += 1
        else:
            j += 1
    return resultado


def horas(intervalos):
    return {h for inicio, fin in intervalos for h in range(inicio, fin + 1)}


def intervalos_aleatorios():
    cortes = sorted(random.sample(range(25), 2 * random.randint(0, 5)))
    return [list(par) for par in zip(cortes[::2], cortes[1::2])]


ana = [[8, 10], [12, 15], [17, 20]]
luis = [[9, 13], [15, 18], [19, 22]]
print(interseccion(ana, luis))
random.seed(10)
for _ in range(1000):
    a, b = intervalos_aleatorios(), intervalos_aleatorios()
    assert horas(interseccion(a, b)) == horas(a) & horas(b)
    assert interseccion(a, b) == interseccion(b, a)

[[9, 10], [12, 13], [15, 15], [17, 18], [19, 20]]


## 11. Profundidad de una lista anidada

Escribir una función recursiva `profundidad(x)`, que devuelva 0 si `x` no es una lista y, si lo es, 1 más la mayor profundidad de sus elementos (`[]` tiene profundidad 1). Usar `max` con `default` en lugar de casos especiales y comprobarla con una versión iterativa con pila sobre listas anidadas aleatorias.

In [12]:
def profundidad(x):
    if not isinstance(x, list):
        return 0
    return 1 + max(map(profundidad, x), default=0)


def profundidad_pila(x):
    maximo, pila = 0, [(x, 0)]
    while pila:
        elemento, nivel = pila.pop()
        if isinstance(elemento, list):
            maximo = max(maximo, nivel + 1)
            pila.extend((e, nivel + 1) for e in elemento)
    return maximo


def anidada_aleatoria(nivel=0):
    if nivel > 5 or random.random() < 0.3:
        return random.randint(0, 9)
    return [anidada_aleatoria(nivel + 1) for _ in range(random.randint(0, 3))]


for x in (7, [], [1, 2], [1, [2, [3, []]], 4], [[[[]]]]):
    print(f"{str(x):<22} {profundidad(x)}")
random.seed(11)
for _ in range(1000):
    x = anidada_aleatoria()
    assert profundidad(x) == profundidad_pila(x)

7                      0
[]                     1
[1, 2]                 1
[1, [2, [3, []]], 4]   4
[[[[]]]]               4


## 12. Perfectos, abundantes y deficientes con comprensiones

Un número es perfecto si la suma de sus divisores propios es igual a él, abundante si la supera y deficiente si no llega. Solo con comprensiones: calcular la suma de divisores propios recorriendo hasta $\sqrt n$, agrupar los números hasta 10 000 en un diccionario `tipo → lista`, comprobar que los perfectos son los que da la fórmula de Euclides-Euler $2^{p-1}(2^p - 1)$ con $2^p - 1$ primo, y encontrar con un conjunto por comprensión el mayor número par que **no** es suma de dos abundantes.

In [13]:
from collections import Counter


def suma_divisores_propios(n):
    if n == 1:
        return 0
    return 1 + sum(d + (n // d if d * d != n else 0) for d in range(2, math.isqrt(n) + 1) if n % d == 0)


def tipo(n):
    s = suma_divisores_propios(n)
    return "perfecto" if s == n else "abundante" if s > n else "deficiente"


def es_primo(n):
    return n > 1 and all(n % d for d in range(2, math.isqrt(n) + 1))


N = 10_000
tipos = {n: tipo(n) for n in range(1, N + 1)}
por_tipo = {t: [n for n in tipos if tipos[n] == t] for t in ("perfecto", "abundante", "deficiente")}
print({t: len(ns) for t, ns in por_tipo.items()})
print("Perfectos:", por_tipo["perfecto"])

euclides_euler = [2 ** (p - 1) * (2**p - 1) for p in range(2, 8) if es_primo(2**p - 1)]
assert [n for n in euclides_euler if n <= N] == por_tipo["perfecto"]

abundantes = por_tipo["abundante"]
sumas = {a + b for i, a in enumerate(abundantes) for b in abundantes[i:] if a + b <= N}
pares_sin_suma = [n for n in range(2, N + 1, 2) if n not in sumas]
print("Mayor par que no es suma de dos abundantes:", max(pares_sin_suma))
assert all(suma_divisores_propios(n) == sum(d for d in range(1, n) if n % d == 0) for n in range(1, 2000))
assert all(any(n - a in set(abundantes) for a in abundantes if a < n) for n in range(max(pares_sin_suma) + 2, 2001, 2))

{'perfecto': 4, 'abundante': 2488, 'deficiente': 7508}
Perfectos: [6, 28, 496, 8128]
Mayor par que no es suma de dos abundantes: 46


## 13. Aplanar con `chain` y agrupar con `groupby`

Un diccionario guarda, por isla, las farmacias de guardia de una semana como lista de listas (una por día). Aplanarlo con `itertools.chain.from_iterable` y agrupar las guardias por municipio con `groupby`. Mostrar que `groupby` solo junta elementos **consecutivos** con la misma clave, así que hay que ordenar antes, y comprobar el resultado con un `Counter`.

In [14]:
from itertools import chain

guardias = {
    "Tenerife": [["La Laguna", "Arona"], ["Santa Cruz", "La Laguna"], ["Arona", "Adeje"]],
    "La Palma": [["Los Llanos"], ["Santa Cruz de La Palma", "Los Llanos"]],
    "El Hierro": [["Valverde"], ["Frontera"]],
}
todas = list(chain.from_iterable(chain.from_iterable(guardias.values())))
print(len(todas), "guardias:", todas)

sin_ordenar = [(m, len(list(g))) for m, g in groupby(todas)]
ordenadas = [(m, len(list(g))) for m, g in groupby(sorted(todas))]
print("Sin ordenar:", sin_ordenar[:4], "...")
print("Ordenadas:  ", ordenadas)

assert len(sin_ordenar) > len(ordenadas)
assert dict(ordenadas) == Counter(todas)
assert todas == [m for isla in guardias.values() for dia in isla for m in dia]

11 guardias: ['La Laguna', 'Arona', 'Santa Cruz', 'La Laguna', 'Arona', 'Adeje', 'Los Llanos', 'Santa Cruz de La Palma', 'Los Llanos', 'Valverde', 'Frontera']
Sin ordenar: [('La Laguna', 1), ('Arona', 1), ('Santa Cruz', 1), ('La Laguna', 1)] ...
Ordenadas:   [('Adeje', 1), ('Arona', 2), ('Frontera', 1), ('La Laguna', 2), ('Los Llanos', 2), ('Santa Cruz', 1), ('Santa Cruz de La Palma', 1), ('Valverde', 1)]
